3. Обучение `yolov8n-obb` (быстрая модель).
4. Обучение `yolo11l-obb` (точная модель).

## 1. Окружение

In [11]:
# !pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu132

Looking in indexes: https://download.pytorch.org/whl/cu132
   ---------------------------------------- 0.0/1.9 GB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 GB 42.8 MB/s eta 0:00:46
   ---------------------------------------- 0.0/1.9 GB 47.1 MB/s eta 0:00:41
    --------------------------------------- 0.0/1.9 GB 56.7 MB/s eta 0:00:34
    --------------------------------------- 0.0/1.9 GB 57.2 MB/s eta 0:00:34
   - -------------------------------------- 0.1/1.9 GB 59.8 MB/s eta 0:00:32
   - -------------------------------------- 0.1/1.9 GB 61.8 MB/s eta 0:00:31
   - -------------------------------------- 0.1/1.9 GB 62.6 MB/s eta 0:00:30
   -- ------------------------------------- 0.1/1.9 GB 63.0 MB/s eta 0:00:30
   -- ------------------------------------- 0.1/1.9 GB 63.3 MB/s eta 0:00:29
   -- ------------------------------------- 0.1/1.9 GB 63.5 MB/s eta 0:00:29
   -- ------------------------------------- 0.1/1.9 GB 63.3 MB/s eta 0:00:29
   --- ------------------

In [15]:
# !pip list torch

Package                   Version
------------------------- ------------
anyio                     4.13.0
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
arrow                     1.4.0
asttokens                 3.0.1
async-lru                 2.3.0
attrs                     26.1.0
babel                     2.18.0
beautifulsoup4            4.15.0
bleach                    6.4.0
certifi                   2026.5.20
cffi                      2.0.0
charset-normalizer        3.4.7
colorama                  0.4.6
comm                      0.2.3
contourpy                 1.3.3
cycler                    0.12.1
debugpy                   1.8.21
decorator                 5.3.1
defusedxml                0.7.1
executing                 2.2.1
fastjsonschema            2.21.2
filelock                  3.29.4
fonttools                 4.63.0
fqdn                      1.5.1
fsspec                    2026.4.0
h11                       0.16.0
httpcore                  1.0.9
httpx       

In [3]:
print("test")
torch.cuda.is_available()

test


True

In [2]:
import sys, subprocess, os, json, shutil, random
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROJECT_ROOT = PROJECT_ROOT.resolve()
print("Project root:", PROJECT_ROOT)

# pip install: ultralytics, opencv-python, pyyaml — для ноутбука.
def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ("ultralytics>=8.2", "opencv-python", "pyyaml", "torchvision"):
    try:
        __import__(pkg.split(">=")[0])
    except ImportError:
        pip_install(pkg)

import torch, yaml
from ultralytics import YOLO

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE)
print("torch:", torch.__version__)

Project root: C:\Users\popop\OneDrive\Documents\python\isolator-detector
Device: cuda
torch: 2.12.0+cu132


In [4]:
torch.zeros(1).cuda()

tensor([0.], device='cuda:0')

## 2. Подготовка датасета

Запускаем `scripts/prepare_dataset.py`. По умолчанию собираем **весь** датасет (7988 изображений).

Для быстрого прогона можно выставить `MAX_IMAGES=600` — этого хватит, чтобы проверить, что пайплайн работает, но для финального обучения лучше использовать полный датасет.

In [8]:
MAX_IMAGES = 0   # 0 = использовать все 7988 изображений. Поставьте 600 для быстрого прогона.
VAL_FRAC = 0.15
USE_SYMLINKS = True

# Пути передаём абсолютные — иначе subprocess стартует из CWD ноутбука
# (обычно `notebooks/`) и относительный путь ломается.
SOURCE_DIR = PROJECT_ROOT / "Дефекты линий электропередач_v3" / "insulators"
assert SOURCE_DIR.exists(), f"Датасет не найден: {SOURCE_DIR}"
DATA_DIR = (PROJECT_ROOT / "data" / "insulators_yolo").resolve()
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)

cmd = [
    sys.executable, str(PROJECT_ROOT / "scripts" / "prepare_dataset.py"),
    "--source", str(SOURCE_DIR),
    "--out",    str(DATA_DIR),
    "--val-frac", str(VAL_FRAC),
]
if MAX_IMAGES:
    cmd += ["--max-images", str(MAX_IMAGES)]
if USE_SYMLINKS:
    cmd.append("--use-symlinks")
print(" ", " ".join(cmd))
# Запускаем из PROJECT_ROOT, чтобы любые относительные пути внутри скрипта
# резолвились от корня репозитория.
subprocess.check_call(cmd, cwd=str(PROJECT_ROOT))

print("\ndata.yaml:")
print((DATA_DIR / "data.yaml").read_text())

  C:\Users\popop\AppData\Local\Python\pythoncore-3.14-64\python.exe C:\Users\popop\OneDrive\Documents\python\isolator-detector\scripts\prepare_dataset.py --source C:\Users\popop\OneDrive\Documents\python\isolator-detector\Дефекты линий электропередач_v3\insulators --out C:\Users\popop\OneDrive\Documents\python\isolator-detector\data\insulators_yolo --val-frac 0.15 --use-symlinks

data.yaml:
path: C:\Users\popop\OneDrive\Documents\python\isolator-detector\data\insulators_yolo
train: images/train
val: images/val

nc: 8
names:
  0: vibration_damper
  1: festoon_insulators
  2: polymer_insulators
  3: traverse
  4: nest
  5: bad_insulator
  6: damaged_insulator
  7: safety_sign+



## 3. Обучение быстрой модели: YOLOv8n-OBB

Nano OBB-модель — т.к нужно выкатить детекцию на слабую машину или в реал-тайм сценарий. Жертвуем точностью ради скорости, но за счёт OBB всё равно держим mAP выше, чем у HBB-аналога того же размера.

In [9]:
FAST_MODEL_NAME = "yolov8n-obb"
FAST_EPOCHS = 60 if MAX_IMAGES == 0 else 15
FAST_IMGSZ = 960
FAST_BATCH = 12  # imgsz 960 — снижаем батч, чтобы влезть в RAM/MPS
fast_model = YOLO(f"{FAST_MODEL_NAME}.pt")

In [10]:
fast_results = fast_model.train(
    data=str(DATA_DIR / "data.yaml"),
    epochs=FAST_EPOCHS,
    imgsz=FAST_IMGSZ,
    batch=FAST_BATCH,
    device=DEVICE,
    project=str(PROJECT_ROOT / "runs"),
    name=f"{FAST_MODEL_NAME}_insulators",
    exist_ok=True,
    patience=15,
    workers=4,
    cache=False,
    verbose=True,
    seed=42,
    # Color
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    # Geometry — дрон-съёмка: повороты, перспектива, зеркалирование
    degrees=10.0, translate=0.15, scale=0.5, shear=2.0,
    perspective=0.0005, fliplr=0.5, flipud=0.1,
    # Composition
    mosaic=1.0, mixup=0.15, copy_paste=0.2,
    # Occlusion
    erasing=0.2,
)

Ultralytics 8.4.67  Python-3.14.6 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\popop\OneDrive\Documents\python\isolator-detector\data\insulators_yolo\data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.2, exist_ok=True, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolov8n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n-obb_insulator

In [11]:
fast_metrics = fast_model.val(data=str(DATA_DIR / "data.yaml"), imgsz=FAST_IMGSZ, device=DEVICE, verbose=False)
# OBBMetrics.mean_results() -> [precision, recall, mAP@0.5, mAP@0.5:0.95]
_, _, fast_map50, fast_map50_95 = (float(x) for x in fast_metrics.mean_results())
print(f"[{FAST_MODEL_NAME}] mAP@0.5 = {fast_map50:.4f}  mAP@0.5:0.95 = {fast_map50_95:.4f}")

Ultralytics 8.4.67  Python-3.14.6 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
YOLOv8n-obb summary (fused): 82 layers, 3,078,779 parameters, 0 gradients, 8.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1413.51246.4 MB/s, size: 417.8 KB)
val: Scanning C:\Users\popop\OneDrive\Documents\python\isolator-detector\data\insulators_yolo\labels\val.cache... 1198 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1198/1198 502.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 75/75 4.3it/s 17.4s0.2s
                   all       1198       7004      0.874      0.827      0.872      0.687
Speed: 4.9ms preprocess, 3.2ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to C:\Users\popop\OneDrive\Documents\python\isolator-detector\notebooks\runs\obb\val
[yolov8n-obb] mAP@0.5 = 0.8717  mAP@0.5:0.95 = 0.6865


## 4. Обучение точной модели: YOLO11l-OBB

Large OBB-модель (YOLO11) больше параметров и более новая архитектура → выше точность, но и inference в 4–5× медленнее, чем у fast-модели.

In [12]:
ACCURATE_MODEL_NAME = "yolo11l-obb"
ACC_EPOCHS = 60 if MAX_IMAGES == 0 else 20
ACC_IMGSZ = 960
ACC_BATCH = 4
acc_model = YOLO(f"{ACCURATE_MODEL_NAME}.pt")

In [13]:
acc_results = acc_model.train(
    data=str(DATA_DIR / "data.yaml"),
    epochs=ACC_EPOCHS,
    imgsz=ACC_IMGSZ,
    batch=ACC_BATCH,
    device=DEVICE,
    project=str(PROJECT_ROOT / "runs"),
    name=f"{ACCURATE_MODEL_NAME}_insulators",
    exist_ok=True,
    patience=25,
    workers=2,
    cache=False,
    verbose=True,
    seed=42,
    # Color
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    # Geometry — дрон-съёмка: повороты, перспектива, зеркалирование
    degrees=10.0, translate=0.15, scale=0.5, shear=2.0,
    perspective=0.0005, fliplr=0.5, flipud=0.1,
    # Composition
    mosaic=1.0, mixup=0.15, copy_paste=0.2,
    # Occlusion
    erasing=0.2,
)

Ultralytics 8.4.67  Python-3.14.6 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\popop\OneDrive\Documents\python\isolator-detector\data\insulators_yolo\data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.2, exist_ok=True, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo11l-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11l-obb_insulators

In [14]:
acc_metrics = acc_model.val(data=str(DATA_DIR / "data.yaml"), imgsz=ACC_IMGSZ, device=DEVICE, verbose=False)
# OBBMetrics.mean_results() -> [precision, recall, mAP@0.5, mAP@0.5:0.95]
_, _, acc_map50, acc_map50_95 = (float(x) for x in acc_metrics.mean_results())
print(f"[{ACCURATE_MODEL_NAME}] mAP@0.5 = {acc_map50:.4f}  mAP@0.5:0.95 = {acc_map50_95:.4f}")

Ultralytics 8.4.67  Python-3.14.6 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
YOLO11l-obb summary (fused): 200 layers, 26,133,931 parameters, 0 gradients, 90.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 823.0795.8 MB/s, size: 298.0 KB)
val: Scanning C:\Users\popop\OneDrive\Documents\python\isolator-detector\data\insulators_yolo\labels\val.cache... 1198 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1198/1198 418.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 75/75 1.8it/s 41.8s0.6ss
                   all       1198       7004      0.906      0.873      0.913      0.753
Speed: 3.3ms preprocess, 25.7ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to C:\Users\popop\OneDrive\Documents\python\isolator-detector\notebooks\runs\obb\val-2
[yolo11l-obb] mAP@0.5 = 0.9130  mAP@0.5:0.95 = 0.7527


## 5. Сравнение и сохранение лучших весов в `app/models/`

In [15]:
runs_dir = PROJECT_ROOT / "runs"
fast_best = runs_dir / f"{FAST_MODEL_NAME}_insulators" / "weights" / "best.pt"
acc_best  = runs_dir / f"{ACCURATE_MODEL_NAME}_insulators" / "weights" / "best.pt"

app_models = PROJECT_ROOT / "app" / "models"
app_models.mkdir(parents=True, exist_ok=True)

fast_dst = app_models / "fast_yolov8n-obb.pt"
acc_dst  = app_models / "accurate_yolo11l-obb.pt"

shutil.copy2(fast_best, fast_dst)
shutil.copy2(acc_best, acc_dst)

print("Saved:")
print(" ", fast_dst, fast_dst.stat().st_size // 1024 // 1024, "MB")
print(" ", acc_dst,  acc_dst.stat().st_size // 1024 // 1024,  "MB")

summary = {
    "fast":     {"model": FAST_MODEL_NAME,     "mAP@0.5": fast_map50, "mAP@0.5:0.95": fast_map50_95, "weights": str(fast_dst.name)},
    "accurate": {"model": ACCURATE_MODEL_NAME, "mAP@0.5": acc_map50,  "mAP@0.5:0.95": acc_map50_95,  "weights": str(acc_dst.name)},
}
(app_models / "metrics.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))
print("\nMetrics:", json.dumps(summary, indent=2, ensure_ascii=False))

Saved:
  C:\Users\popop\OneDrive\Documents\python\isolator-detector\app\models\fast_yolov8n-obb.pt 6 MB
  C:\Users\popop\OneDrive\Documents\python\isolator-detector\app\models\accurate_yolo11l-obb.pt 50 MB

Metrics: {
  "fast": {
    "model": "yolov8n-obb",
    "mAP@0.5": 0.8717172887552047,
    "mAP@0.5:0.95": 0.686504834583672,
    "weights": "fast_yolov8n-obb.pt"
  },
  "accurate": {
    "model": "yolo11l-obb",
    "mAP@0.5": 0.913012842635073,
    "mAP@0.5:0.95": 0.7527037219595323,
    "weights": "accurate_yolo11l-obb.pt"
  }
}


In [16]:
# пробуем загрузить обе модели и сделать предсказание на одной val-картинке.
from PIL import Image
val_imgs = list((DATA_DIR / "images" / "val").iterdir())
sample = random.choice(val_imgs)
print("Sample:", sample)

for label, path in [("FAST", fast_dst), ("ACCURATE", acc_dst)]:
    m = YOLO(str(path))
    r = m.predict(source=str(sample), imgsz=960, conf=0.25, verbose=False)[0]
    # OBB-модели: r.obb (повёрнутые рамки). r.boxes для них None.
    n = len(r.obb) if r.obb is not None else 0
    print(f"{label}: {n} OBB boxes")
print("\nГотово. Веса готовы к использованию веб-приложением.")

Sample: C:\Users\popop\OneDrive\Documents\python\isolator-detector\data\insulators_yolo\images\val\T334-18-_JPG_jpg.rf.fa3cb5000b9d550d47677769c935cb3a.jpg
FAST: 8 OBB boxes
ACCURATE: 10 OBB boxes

Готово. Веса готовы к использованию веб-приложением.
